# K-Nearest Neighbours Example (Wine Quality Dataset)

**Goal: Classify wine quality tier and predict raw quality score, sweeping k from 1 to 20 to find the optimal value.**

## 1. Setup and Data Loading

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys, os

# NOTEBOOK_DIR resolves to the folder this notebook lives in.
# Jupyter sets the working directory to wherever it was launched from,
# so we use __file__ would not work — os.path.abspath('') is the correct
# approach for Jupyter notebooks.
# Find the repo root reliably on any machine.
# We search upward from the current working directory until we find
# the 'data' folder, which only exists at the repo root.
def _find_repo_root():
    path = os.path.abspath(os.getcwd())
    for _ in range(10):  # search up to 10 levels up
        if os.path.isdir(os.path.join(path, 'data')) and os.path.isdir(os.path.join(path, 'src')):
            return path
        path = os.path.dirname(path)
    raise RuntimeError(
        "Could not find repo root. Make sure you launched Jupyter from inside the CMOR-438 folder."
    )

REPO_ROOT = _find_repo_root()

# Algorithm source files live in src/supervised/ at the repo root,
# which is four levels up from examples/supervised/<algo>/
SRC_SUP  = os.path.join(REPO_ROOT, 'src', 'supervised')
sys.path.insert(0, SRC_SUP)

# Data files live in data/ at the repo root
DATA_DIR = os.path.join(REPO_ROOT, 'data')
from k_nearest_neighbors import KNNClassifier, KNNRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

wine = pd.read_csv(os.path.join(DATA_DIR, 'WineQT.csv')).drop(columns=['Id'])
FEATURE_COLS = [c for c in wine.columns if c != 'quality']
print(f"Dataset loaded: {wine.shape[0]} samples, {len(FEATURE_COLS)} features.")

## 2. Preprocessing

In [ ]:
X = StandardScaler().fit_transform(wine[FEATURE_COLS].values.astype(float))
y_reg = wine['quality'].values.astype(float)
y_clf = np.array([0 if q<=4 else (1 if q<=6 else 2) for q in y_reg])

X_tr, X_te, y_tr_clf, y_te_clf = train_test_split(X, y_clf, test_size=0.2, random_state=42, stratify=y_clf)
_, __, y_tr_reg, y_te_reg = train_test_split(X, y_reg, test_size=0.2, random_state=42)
print(f"Training samples: {X_tr.shape[0]}  |  Test samples: {X_te.shape[0]}")

## 3. Sweep k Values

In [ ]:
k_range = range(1, 21)
clf_accs, reg_r2s = [], []
for k in k_range:
    kc = KNNClassifier(k=k, weights='distance').fit(X_tr, y_tr_clf)
    clf_accs.append(kc.accuracy(X_te, y_te_clf))
    kr = KNNRegressor(k=k, weights='distance').fit(X_tr, y_tr_reg)
    reg_r2s.append(kr.score(X_te, y_te_reg))

best_k_clf = list(k_range)[np.argmax(clf_accs)]
best_k_reg = list(k_range)[np.argmax(reg_r2s)]
print(f'Best K (Classifier): {best_k_clf}  Accuracy={max(clf_accs):.4f}')
print(f'Best K (Regressor):  {best_k_reg}  R²={max(reg_r2s):.4f}')

## 4. Results and Visualisation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(list(k_range), clf_accs, 'o-', color='steelblue', lw=1.5, ms=5)
axes[0].axvline(best_k_clf, color='red', linestyle='--', lw=1.2, label=f'Best k={best_k_clf}')
axes[0].set_xlabel('k'); axes[0].set_ylabel('Accuracy')
axes[0].set_title('KNN Classifier - Accuracy vs k', fontweight='bold'); axes[0].legend()

axes[1].plot(list(k_range), reg_r2s, 'o-', color='darkorange', lw=1.5, ms=5)
axes[1].axvline(best_k_reg, color='red', linestyle='--', lw=1.2, label=f'Best k={best_k_reg}')
axes[1].set_xlabel('k'); axes[1].set_ylabel('R²')
axes[1].set_title('KNN Regressor - R² vs k', fontweight='bold'); axes[1].legend()
plt.tight_layout(); plt.show()

## 5. Analysis

**Classifier result: Best k=7, Accuracy=90.4%** — the highest classification accuracy of all models.

**Regressor result: Best k=16, R²=-0.41** — a negative R², meaning the KNN regressor is worse than simply predicting the mean quality score for every wine.

These two results tell a striking and coherent story:

**Why classification succeeds:** The 3-class tiering (Low/Mid/High) smooths over the fine-grained quality differences that are hard to predict precisely. With k=7 and distance weighting, the model effectively asks "what do the 7 most similar wines in the training set look like, and what tier are they in?" This works because similar physicochemical profiles genuinely cluster by quality tier.

**Why regression fails:** Predicting the exact quality score (3, 4, 5, 6, 7, or 8) is a much harder problem. The KNN regressor averages the k nearest neighbours' scores, but this averaging works against it — the neighbours of a score-7 wine often include score-5 and score-6 wines, pulling the prediction toward the mean. The negative R² confirms that averaging is actively harmful here — the variance within neighbourhoods is too high.

**The k-sweep plots** illustrate the bias-variance tradeoff clearly. For the classifier, accuracy rises steeply from k=1 (overfit) to around k=7 (sweet spot) then gradually declines as k increases (underfit). The regressor shows a similar shape but the best R² is still negative, confirming the task itself is too noisy for KNN regression.

**Key takeaway:** KNN is a powerful classifier here but a poor regressor for this dataset. The lesson is that classification (which tier?) is a fundamentally easier problem than regression (which exact score?) when the target has high within-neighbourhood variance.